# Confidence intervals

Core stage, lesson C7. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** compute an interval for a share and for a mean, read it as long-run coverage, bootstrap an interval for a median, tell a confidence interval from a prediction interval, and write the project report of Part 1.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 and of the project step start empty: you write them.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3', 'Project step']


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want, tol)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)

# One number for each order: the sum of its lines, quantity × unit price − discount.
# All 1,800 orders count: the canceled order and order 2075 (the line with the extra zero) are in.
price = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(amount=order_items["quantity"] * order_items["product_id"].map(price) - order_items["discount_usd"])
orders["value"] = orders["order_id"].map(lines.groupby("order_id")["amount"].sum())
ordered = pd.to_datetime(orders["ordered_at"])
orders["hour"] = ordered.dt.hour
orders["weekday"] = ordered.dt.day_name()
assert orders["value"].notna().all() and round(orders["value"].sum(), 2) == 22754.25
print(f"Data loaded: {len(orders):,} orders, {orders['value'].mean():.2f} dollars on average, {orders['value'].sum():,.2f} dollars in all.")

# The population of this lesson: the value of each of the 1,800 orders, as an array, and its mean.
values = orders["value"].to_numpy()
mu = values.mean()

## Unit 1: Estimate plus and minus a margin

**Predict first.** A sample of 200 orders gives a share of mobile payments. Will the interval around it contain the share of all 1,800 orders, printed last? And how wide will it be: a few percentage points or tens?

In [ ]:
rng = np.random.default_rng(42)
pick = rng.integers(0, len(orders), size=200)             # 200 order positions, drawn with replacement
sample = orders.iloc[pick]                                # the sample of 200 orders
n = len(sample)
p_hat = (sample["payment_method"] == "mobile").mean()     # the share paid by mobile in the sample
se = np.sqrt(p_hat * (1 - p_hat) / n)                     # the standard error of a share
print(n, round(float(p_hat), 3), round(float(se), 3))
print(round(float(p_hat - 1.96 * se), 3), round(float(p_hat + 1.96 * se), 3))
print(round(float((orders["payment_method"] == "mobile").mean()), 3))   # the truth for all 1,800 orders

**Predict first.** For a mean, the multiplier is `t*` instead of 1.96. Will `t*` for 200 orders be above or below 1.96? Will SciPy's interval agree with the hand-made one, and will it contain the true mean, printed last?

In [ ]:
x = sample["value"].to_numpy()                    # the 200 order values
m, s = x.mean(), x.std(ddof=1)
sem = s / np.sqrt(n)                              # the standard error of the mean, estimated from the sample
t_star = stats.t.ppf(0.975, df=n - 1)             # the t multiplier for 95 percent
print(round(float(m), 2), round(float(sem), 2), round(float(t_star), 3))
print(round(float(m - t_star * sem), 2), round(float(m + t_star * sem), 2))
low, high = stats.t.interval(0.95, df=n - 1, loc=m, scale=sem)    # the same interval from SciPy
print(round(float(low), 2), round(float(high), 2), round(float(mu), 2))

## Unit 2: What the 95 percent means, and what makes an interval narrower

**Predict first.** The cell builds 1,000 intervals from 1,000 samples of 200 orders. How many will contain the true mean? Of those that miss it, will more lie below it or above it?

In [ ]:
rng = np.random.default_rng(42)
low_miss = high_miss = 0
spans = []                                        # the first 20 intervals, for the picture
for k in range(1000):
    xs = values[rng.integers(0, len(values), size=200)]
    lo, hi = stats.t.interval(0.95, df=199, loc=xs.mean(), scale=xs.std(ddof=1) / np.sqrt(200))
    low_miss += hi < mu                           # the interval lies below the true mean
    high_miss += lo > mu                          # the interval lies above it
    if k < 20:
        spans.append((float(lo), float(hi)))
print(1000 - low_miss - high_miss, low_miss, high_miss)
print(sum(1 for lo, hi in spans if lo <= mu <= hi), "of the first 20 contain the true mean")

**Predict first.** The cell prints the width of the interval for three sample sizes (50, 200, 800) at three confidence levels (90, 95, 99 percent). What happens to the width when n grows four times, and when the level goes up?

In [ ]:
s_all = values.std(ddof=1)                        # the spread of single orders, kept fixed
for size in (50, 200, 800):
    widths = [2 * stats.t.ppf(0.5 + level / 2, df=size - 1) * s_all / np.sqrt(size) for level in (0.90, 0.95, 0.99)]
    print(size, [round(float(w), 2) for w in widths])

## Unit 3: A bootstrap interval for a median

**Predict first.** The cell resamples the 200 orders of unit 1 2,000 times and takes the median of each resample. Will the interval contain the sample's median? Will it contain the median of all 1,800 orders, printed last?

In [ ]:
rng = np.random.default_rng(7)
boot = np.array([np.median(rng.choice(x, size=len(x))) for _ in range(2000)])    # 2,000 resamples of the sample, with replacement
print(round(float(np.median(x)), 2), round(float(boot.std(ddof=1)), 2))
b_low, b_high = np.percentile(boot, [2.5, 97.5])                                 # the middle 95 percent of the 2,000 medians
print(round(float(b_low), 2), round(float(b_high), 2), round(float(np.median(values)), 2))
res = stats.bootstrap((x,), np.median, confidence_level=0.95, method="percentile", rng=np.random.default_rng(7))
print(round(float(res.confidence_interval.low), 2), round(float(res.confidence_interval.high), 2))

## Unit 4: An interval for the mean and an interval for the next order

**Predict first.** The interval for the mean of the sample was 11.80 to 13.81 dollars. How much wider will an interval for one single new order be? Can the formula's lower end be a real order value, when the cheapest order of all is printed last?

In [ ]:
half = t_star * s * np.sqrt(1 + 1 / n)            # the bell-shape formula for the next single order
pi_low, pi_high = m - half, m + half
q_low, q_high = np.percentile(x, [2.5, 97.5])     # the middle 95 percent of the sample itself
print(round(float(pi_low), 2), round(float(pi_high), 2))
print(round(float(q_low), 2), round(float(q_high), 2))
share = lambda lo, hi: round(float(((values >= lo) & (values <= hi)).mean()), 3)    # the share of all 1,800 orders inside
print(share(pi_low, pi_high), share(q_low, q_high), float(values.min()))

## Finish it

### Exercise 1: Finish it

**Task:** What share of Harbor's orders is paid by mobile, give or take? Take the 355 Harbor orders as the sample. Compute the standard error of the share and the 95 percent interval p_hat plus and minus 1.96 standard errors, each end rounded to 3 decimals, as a list [low, high].

**Expected:** `answer_1` is a list of two numbers.

In [ ]:
harbor = orders[orders["store_id"] == 3]                       # given: the 355 Harbor orders
p_h = (harbor["payment_method"] == "mobile").mean()           # given: the share paid by mobile
# the standard error of the share, then the interval, each end rounded to 3 decimals
answer_1 = None  # replace None with [low, high]

In [ ]:
_ = check("Exercise 1", "answer_1", [0.289, 0.387], traps={(0.337, 0.339): 'That interval is far too narrow: the standard error of a share has a square root, sqrt(p * (1 - p) / n).'})

<details>
<summary>Hint</summary>

Unit 1 used `np.sqrt(p_hat * (1 - p_hat) / n)` for the standard error and `p_hat - 1.96 * se`, `p_hat + 1.96 * se` for the two ends.

</details>

<details>
<summary>Solution</summary>

```python
harbor = orders[orders["store_id"] == 3]
p_h = (harbor["payment_method"] == "mobile").mean()
se_h = np.sqrt(p_h * (1 - p_h) / len(harbor))
answer_1 = [round(float(p_h - 1.96 * se_h), 3), round(float(p_h + 1.96 * se_h), 3)]
```

120 of Harbor's 355 orders are mobile, a share of 0.338, with a standard error of 0.025: the interval is 0.289 to 0.387. Without the square root the interval would be 0.337 to 0.339, far too narrow, and the check cell says so.

</details>

## On your own

### Exercise 2: On your own

**Task:** What is the mean order value at Campus, give or take? Before you start, say in one sentence what the result looks like: two dollar amounts, a low and a high. Take the Campus orders, store 2, as the sample and build the 95 percent interval for their mean with the t multiplier, each end rounded to 2 decimals, as a list [low, high]. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [12.0, 13.15], traps={(3.25, 31.28): 'These numbers are the range of single Campus orders. The interval for the mean is far narrower: divide the SD by the square root of n.'})

<details>
<summary>Hint</summary>

Get the values of the Campus orders as an array, then repeat unit 1: the SD with `ddof=1`, divided by the square root of n, and `stats.t.interval(0.95, df=n - 1, loc=mean, scale=sem)`.

</details>

<details>
<summary>Solution</summary>

```python
campus = orders[orders["store_id"] == 2]["value"].to_numpy()
sem_c = campus.std(ddof=1) / np.sqrt(len(campus))
low_c, high_c = stats.t.interval(0.95, df=len(campus) - 1, loc=campus.mean(), scale=sem_c)
answer_2 = [round(float(low_c), 2), round(float(high_c), 2)]
```

12.0 to 13.15 dollars: the 716 Campus orders average 12.58 with a standard error of 0.29. The middle 95 percent of single Campus orders is a much wider range, 3.25 to 31.28, and it answers another question: where one order falls, not where the mean is. The check cell reports those two numbers as a hint.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague bootstraps the median order value of a sample of 100 Old Town orders, but the interval comes out with no width: both ends are the same number. Fix the lines, then set `answer_3` to the 95 percent bootstrap interval, each end rounded to 2 decimals, as a list [low, high].

**Expected:** `answer_3` is close to `[8.0, 13.75]`: the check cell accepts a difference of up to 0.75 in each end, because the numbers depend on the seed.

In [ ]:
rng = np.random.default_rng(7)
old_town = orders[orders["store_id"] == 1]["value"].to_numpy()
sample_3 = rng.choice(old_town, size=100, replace=False)       # the sample of 100 Old Town orders
boot_3 = np.array([np.median(rng.choice(sample_3, size=100, replace=False)) for _ in range(2000)])   # the colleague's resamples
answer_3 = [round(float(v), 2) for v in np.percentile(boot_3, [2.5, 97.5])]

In [ ]:
_ = check("Exercise 3", "answer_3", [8.0, 13.75], traps={(9.62, 9.62): 'The interval has no width. Every resample must be able to repeat some orders and leave others out.'}, tol=0.75)

<details>
<summary>Hint</summary>

Print the first five of `boot_3`. A resample of 100 values drawn from 100 values without replacement holds exactly the same 100 values. What does `replace` do?

</details>

<details>
<summary>Solution</summary>

```python
rng = np.random.default_rng(7)
old_town = orders[orders["store_id"] == 1]["value"].to_numpy()
sample_3 = rng.choice(old_town, size=100, replace=False)
boot_3 = np.array([np.median(rng.choice(sample_3, size=100)) for _ in range(2000)])
answer_3 = [round(float(v), 2) for v in np.percentile(boot_3, [2.5, 97.5])]
```

The first draw, the sample, is rightly without replacement: no order twice. The resamples must be with replacement. Without it, every resample is the same 100 orders in another order, so every median is the same and the interval has no width: the check cell reports one number twice.

</details>

## Project step: a descriptive report on cafe-sales

Part 1 of the advanced core ends here. **The brief.** The owner wants a one-page report on the orders of cafe-sales, January to June 2026, 1,800 orders, to read before a meeting. It answers three questions in plain words: **What is typical? How spread out? How sure are you?** Treat the 1,800 orders as a sample of the orders these cafes take on an ordinary day. Work in the empty cell below, then run the check cell.

Name these variables: `typical_median`, `typical_mean`, `spread_sd`, `spread_iqr` (each rounded to 2 decimals), `ci_mean` (95 percent, t multiplier, 2 decimals, [low, high]), `ci_share` (the 95 percent interval of the share paid by mobile, 3 decimals, [low, high]) and `ci_median` (a 95 percent bootstrap interval of the median, 2 decimals, [low, high]). Then write `report` with an f-string, so that every number in it comes from a variable.

**Self-check list, before you call it done:**

1. Every number in `report` comes from a cell: none is typed.
2. It gives a median and a mean and says which one it would quote for a typical order, and why.
3. It says whether order 2075, the 197.00 order with the extra zero, is in the numbers.
4. It gives the spread two ways: the SD and the IQR.
5. Each interval says its level and reads as a range of plausible values for the parameter, not as a share of orders.
6. One sentence says who the data can speak for and who it cannot.

In [ ]:
_ = check_all("Project step", {'typical_median': (11.75, 0.005), 'typical_mean': (12.64, 0.005), 'spread_sd': (8.62, 0.005), 'spread_iqr': (9.5, 0.005), 'ci_mean': ([12.24, 13.04], 0.005), 'ci_share': ([0.282, 0.325], 0.0005), 'ci_median': ([11.0, 12.25], 0.5)})

<details>
<summary>Hint</summary>

Reuse the cells of units 1 to 4 with all 1,800 orders: `values` for the order values, `np.percentile(values, [25, 75])` for the IQR, `stats.t.interval` for the mean, the share formula of unit 1, and the bootstrap of unit 3 with `size=len(values)`.

</details>

<details>
<summary>A model solution</summary>

```python
typical_median = round(float(np.median(values)), 2)
typical_mean = round(float(values.mean()), 2)
spread_sd = round(float(values.std(ddof=1)), 2)
q1, q3 = np.percentile(values, [25, 75])
spread_iqr = round(float(q3 - q1), 2)

n_all = len(values)
sem_all = values.std(ddof=1) / np.sqrt(n_all)
low, high = stats.t.interval(0.95, df=n_all - 1, loc=values.mean(), scale=sem_all)
ci_mean = [round(float(low), 2), round(float(high), 2)]

share_mobile = float((orders["payment_method"] == "mobile").mean())
se_share = np.sqrt(share_mobile * (1 - share_mobile) / n_all)
ci_share = [round(share_mobile - 1.96 * se_share, 3), round(share_mobile + 1.96 * se_share, 3)]

rng = np.random.default_rng(11)
boot_all = np.array([np.median(rng.choice(values, size=n_all)) for _ in range(2000)])
ci_median = [round(float(v), 2) for v in np.percentile(boot_all, [2.5, 97.5])]

report = (f"The typical order is worth {typical_median:.2f} dollars (the median). The mean, {typical_mean:.2f}, is pulled up by a few large orders, "
          f"one of them {values.max():.2f} dollars (order 2075). "
          f"The middle half of the orders runs from {q1:.2f} to {q3:.2f} dollars (IQR {spread_iqr:.2f}); the SD is {spread_sd:.2f}. "
          f"If these six months are a fair sample of ordinary trade, the mean order is between {ci_mean[0]:.2f} and {ci_mean[1]:.2f} dollars "
          f"and the median between {ci_median[0]:.2f} and {ci_median[1]:.2f} (95% intervals), and {share_mobile:.1%} of orders are paid by mobile, "
          f"between {ci_share[0]:.1%} and {ci_share[1]:.1%}. "
          f"The data cover three cafes from January to June 2026, and Harbor only from April, so they do not speak for other cafes or later months.")
print(report)
```

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")